# Lenguajes y Autómatas II (SCD-1016)
## Sesión 25 · Control de Flujo: Bucles While y For

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Cómo se representa un ciclo que se repite sin saber de antemano cuántas veces?
>
> **Objetivo de la sesión:** Construir el esqueleto TAC de while y for con etiquetas de inicio y fin, y ubicar correctamente la actualización del iterador.

**Entregable:** tus funciones emitir_while() y emitir_for() en artesano_sesion25_c.c, el harness en verde y la bitácora en auditorias/auditoria_sesion25.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)


## 1. ¿Qué vas a construir?

Ambas funciones agregan líneas al arreglo en la posición *n y avanzan ese contador; las etiquetas se toman de *etiquetas. Un argumento NULL en inicio o post significa que esa instrucción no se emite.

Al final de la sesión tendrás: **Dos funciones en C que emiten el esqueleto TAC de un while y de un for, con etiquetas únicas y el iterador en su lugar.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 25](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion25.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion25.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion25.h"

void emitir_while(char lineas[][64], int max, int *n, const char *cond, int *etiquetas);
void emitir_for(char lineas[][64], int max, int *n, const char *inicio, const char *cond, const char *post, int *etiquetas);
```

- El while emite cuatro líneas: marca de inicio, salto condicional, GOTO de regreso y marca de fin.
- El for emite init, marca, salto condicional, post, GOTO y marca de fin; si post es NULL, se omite la actualización.
- Cada ciclo consume dos etiquetas y las agrega a *etiquetas; no reutilices nombres.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion25_c.c` junto con `artesano_sesion25_c.c`
> (o `ia_sesion25_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `while: marca de inicio` | `L0:` |
| 2 | `while: condicion` | `IFFALSE a < b GOTO L1` |
| 3 | `dos ciclos sin colision` | `L2:` |
| 4 | `for: inicializacion` | `i = 0` |
| 5 | `for: actualizacion` | `i = i + 1` |
| 6 | `for sin post (NULL)` | 5 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 25 - Esqueleto TAC de while y for ===
Caso                               esperado:              obtenido:              resultado
[ 1] while: marca de inicio             esperado: L0:          obtenido: L0:          OK
[ 2] while: condicion
     esperado: IFFALSE a < b GOTO L1
     obtenido: IFFALSE a < b GOTO L1
     -> OK
[ 3] dos ciclos sin colision            esperado: L2:          obtenido: L2:          OK
[ 4] for: inicializacion                esperado: i = 0        obtenido: i = 0        OK
[ 5] for: actualizacion                 esperado: i = i + 1    obtenido: i = i + 1    OK
[ 6] for sin post (NULL)                esperado: 5            obtenido: 5            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion25.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] El GOTO de regreso apunta a la marca de inicio y el salto condicional a la marca de fin.
- [ ] El for coloca la actualización del iterador antes del GOTO de regreso.
- [ ] Dos ciclos seguidos usan etiquetas distintas (L0/L1 y L2/L3).


## Paso 0 · Entorno


In [ ]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion25_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [ ]:
%%writefile api_sesion25.h
/* api_sesion25.h - contrato de la Sesion 25
 * Esqueleto TAC de while y for
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION25_H
#define API_SESION25_H

void emitir_while(char lineas[][64], int max, int *n, const char *cond, int *etiquetas);
void emitir_for(char lineas[][64], int max, int *n, const char *inicio, const char *cond, const char *post, int *etiquetas);

#endif /* API_SESION25_H */


In [ ]:
%%writefile artesano_sesion25_c.c
/* artesano_sesion25_c.c - Fase 1: El Artesano (sin IA)
 * Esqueleto TAC de while y for.
 *
 * Implementa lo que declara api_sesion25.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion25.h"
#include <stdio.h>
#include <string.h>

void emitir_while(char lineas[][64], int max, int *n, const char *cond, int *etiquetas) {
          (void)lineas; (void)max; (void)n; (void)cond; (void)etiquetas; /* TODO: cuatro lineas del while */
      }
      
      void emitir_for(char lineas[][64], int max, int *n, const char *inicio, const char *cond, const char *post, int *etiquetas) {
          (void)lineas; (void)max; (void)n; (void)inicio; (void)cond; (void)post; (void)etiquetas; /* TODO: desazucarar */
      }


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion25_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [ ]:
%%writefile ia_sesion25_c.c
/* ia_sesion25_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion25.h. No puede traer main().
 */
#include "api_sesion25.h"

/* TODO: pega aqui la propuesta de la IA */


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [ ]:
%%writefile harness_sesion25_c.c
/* harness_sesion25_c.c - auditoria automatica de la Sesion 25
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion25_c.c artesano_sesion25_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion25.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str(const char *caso, const char *esperado,
                          const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12s obtenido: %-12s %s\n",
           probadas, caso, esperado, obtenido ? obtenido : "(NULL)",
           ok ? "OK" : "FALLA");
}

static void verificar_str_largo(const char *caso, const char *esperado,
                                const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %s\n", probadas, caso);
    printf("     esperado: %s\n", esperado);
    printf("     obtenido: %s\n", obtenido ? obtenido : "(NULL)");
    printf("     -> %s\n", ok ? "OK" : "FALLA");
}

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 25 - Esqueleto TAC de while y for ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { char l[16][64] = {{0}}; int n = 0, e = 0; emitir_while(l, 16, &n, "a < b", &e); verificar_str("while: marca de inicio", "L0:", l[0]); }
    { char l[16][64] = {{0}}; int n = 0, e = 0; emitir_while(l, 16, &n, "a < b", &e); verificar_str_largo("while: condicion", "IFFALSE a < b GOTO L1", l[1]); }
    { char l[16][64] = {{0}}; int n = 0, e = 0; emitir_while(l, 16, &n, "a < b", &e); emitir_while(l, 16, &n, "c < d", &e); verificar_str("dos ciclos sin colision", "L2:", l[4]); }
    { char l[16][64] = {{0}}; int n = 0, e = 0; emitir_for(l, 16, &n, "i = 0", "i < 10", "i = i + 1", &e); verificar_str("for: inicializacion", "i = 0", l[0]); }
    { char l[16][64] = {{0}}; int n = 0, e = 0; emitir_for(l, 16, &n, "i = 0", "i < 10", "i = i + 1", &e); verificar_str("for: actualizacion", "i = i + 1", l[3]); }
    { char l[16][64] = {{0}}; int n = 0, e = 0; emitir_for(l, 16, &n, "j = 0", "j < 3", NULL, &e); verificar_int("for sin post (NULL)", (long)(5), (long)(n)); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion25.md" "\n");
    return 0;
}


In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion25_c.c artesano_sesion25_c.c -o test_artesano && ./test_artesano

In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion25_c.c ia_sesion25_c.c -o test_ia && ./test_ia

## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion25.md` a `auditorias/auditoria_sesion25.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
